# ROTBOTS - Full Pipeline
## Topic to Video in One Notebook

All 9 pipeline stages. Run cells top to bottom.

Every cell is an automated decision. What does the machine choose?

---

In [ ]:
# SETUP
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper
import json, re, random, requests, subprocess, shutil, os, time as _time, threading, edge_tts
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML, Audio, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)

def progress_bar(c, t, label='', w=30):
    pct = c / t if t > 0 else 0
    filled = int(w * pct)
    bar = chr(9608) * filled + chr(9617) * (w - filled)
    return f'{bar} {c}/{t} {label} ({pct:.0%})'

def progress_msg(title, c, t, label='', detail=''):
    msg = f'{title} | {progress_bar(c, t, label)}'
    if detail: msg += f' | {detail}'
    return msg

print('[OK] Setup complete')

In [ ]:
# API KEYS
GROQ_API_KEY = ''  # https://console.groq.com/keys
FAL_API_KEY = ''   # https://fal.ai/dashboard/keys
LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'
if GROQ_API_KEY: print('[OK] Groq')
else: print('[!!] Paste GROQ_API_KEY above')
if FAL_API_KEY: os.environ['FAL_KEY'] = FAL_API_KEY; print('[OK] fal.ai')
else: print('[!!] Paste FAL_API_KEY above')

---
# 01: Video Plan

In [ ]:
TOPIC = 'The history and ethics of AI-generated art'
SOURCES = ['https://en.wikipedia.org/wiki/Artificial_intelligence_art']
TOTAL_VIDEO_LENGTH = 60
SECONDS_PER_SCENE = 5
ARCHIVE_RATIO = 0.0
UPLOAD_RATIO = 0.0
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True
SESSION_NAME = ''

In [ ]:
# CALCULATE PLAN
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
NARRATION_LENGTH = CONTENT_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5

# Scene counts: 0-ratio = 0 scenes, AI gets remainder
NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO) if ARCHIVE_RATIO > 0 else 0
NUM_UPLOAD_SCENES = int(NUM_TOTAL_SCENES * UPLOAD_RATIO) if UPLOAD_RATIO > 0 else 0
NUM_AI_SCENES = max(1, NUM_TOTAL_SCENES - NUM_ARCHIVE_SCENES - NUM_UPLOAD_SCENES)
while NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES > NUM_TOTAL_SCENES:
    if NUM_ARCHIVE_SCENES > 0: NUM_ARCHIVE_SCENES -= 1
    elif NUM_UPLOAD_SCENES > 0: NUM_UPLOAD_SCENES -= 1
    else: break

NUM_CHAPTERS = max(1, min(3, NUM_TOTAL_SCENES // 3))
SECTIONS_PER_CHAPTER = max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS)

# Interleave scene order
scene_types = []
ai_p = 0; ar_p = 0; up_p = 0
for i in range(NUM_TOTAL_SCENES):
    remaining = max(1, NUM_TOTAL_SCENES - i)
    ai_need = (NUM_AI_SCENES - ai_p) / remaining
    ar_need = (NUM_ARCHIVE_SCENES - ar_p) / remaining
    up_need = (NUM_UPLOAD_SCENES - up_p) / remaining
    if ar_need >= ai_need and ar_need >= up_need and ar_p < NUM_ARCHIVE_SCENES:
        scene_types.append('archive'); ar_p += 1
    elif up_need >= ai_need and up_p < NUM_UPLOAD_SCENES:
        scene_types.append('upload'); up_p += 1
    else:
        scene_types.append('ai_generated'); ai_p += 1

if not SESSION_NAME:
    SESSION_NAME = '-'.join(re.sub(r'[^a-zA-Z0-9 ]', '', TOPIC.lower()).split()[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['', 'videos', 'audio', 'archive_clips', 'uploads']:
    (SESSION_DIR / d).mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'

plan = dict(topic=TOPIC, sources=SOURCES, session_name=SESSION_NAME,
    total_video_length=TOTAL_VIDEO_LENGTH, seconds_per_scene=SECONDS_PER_SCENE,
    content_length=CONTENT_LENGTH, credits_length=CREDITS_LENGTH,
    narration_length=NARRATION_LENGTH, ai_ratio=AI_RATIO,
    archive_ratio=ARCHIVE_RATIO, upload_ratio=UPLOAD_RATIO,
    num_total_scenes=NUM_TOTAL_SCENES, num_ai_scenes=NUM_AI_SCENES,
    num_archive_scenes=NUM_ARCHIVE_SCENES, num_upload_scenes=NUM_UPLOAD_SCENES,
    scene_types=scene_types, enable_credits=ENABLE_CREDITS,
    enable_subtitles=ENABLE_SUBTITLES, enable_music=ENABLE_MUSIC,
    enable_effects=ENABLE_EFFECTS)
with open(SESSION_DIR / 'video_plan.json', 'w') as f: json.dump(plan, f, indent=2)
with open(SESSION_DIR / 'session_info.json', 'w') as f: json.dump(dict(name=SESSION_NAME, topic=TOPIC), f, indent=2)

sc_counts = Counter(scene_types)
print(f'Session: {SESSION_NAME}')
print(f'Plan: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'Scenes: {NUM_TOTAL_SCENES} total = {sc_counts.get("ai_generated",0)} AI + {sc_counts.get("archive",0)} archive + {sc_counts.get("upload",0)} upload')
print(f'Order: {scene_types}')

---
# 02: Script Writer

In [ ]:
# HELPERS
def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append(dict(role='system', content=system_prompt))
    messages.append(dict(role='user', content=prompt))
    r = requests.post(GROQ_API_URL,
        headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
        json=dict(model=LLM_MODEL, messages=messages,
                  temperature=temperature or LLM_TEMPERATURE,
                  max_tokens=LLM_MAX_TOKENS), timeout=120)
    r.raise_for_status()
    c = r.json()['choices'][0]['message']['content']
    if '<think>' in c and '</think>' in c:
        c = c.split('</think>')[-1].strip()
    return c

def parse_json_response(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text.strip())
    if '```' in text:
        text = text.split('```')[1]
        if text.startswith('json'): text = text[4:]
    text = text.strip()
    for ch in ['{', '[']:
        idx = text.find(ch)
        if idx > 0: text = text[idx:]; break
    for end in ['}', ']']:
        li = text.rfind(end)
        if li != -1:
            candidate = text[:li+1]
            candidate_clean = re.sub(r',\s*([}\]])', r'\1', candidate)
            for t in [candidate, candidate_clean]:
                try: return json.loads(t, strict=False)
                except: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', text), strict=False)

def fetch_website_text(url, max_chars=10000):
    r = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article', 'main', '[role="main"]', '.content', '#content']:
        main = soup.select_one(sel)
        if main: break
    text = (main or soup.find('body') or soup).get_text(separator=' ', strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

print('[OK] Helpers loaded')

In [ ]:
# SCRAPE + SUMMARIZE
source_texts = []
for i, source in enumerate(SOURCES):
    print(f'Scraping {i+1}/{len(SOURCES)}: {source[:60]}')
    try:
        if source.startswith('http'):
            source_texts.append(dict(source=source[:100], text=fetch_website_text(source)))
        else:
            source_texts.append(dict(source=source[:100], text=source))
    except Exception as e: print(f'  Error: {e}')

summaries = []
for i, src in enumerate(source_texts):
    print(f'Summarizing {i+1}/{len(source_texts)}...')
    s = query_llm(
        'Summarize in 2-3 paragraphs for video essay about "' + TOPIC + '":\n\n' + src['text'][:6000],
        'Be concise.')
    summaries.append(dict(source=src['source'], summary=s))

with open(SESSION_DIR / 'summaries.json', 'w') as f:
    json.dump(dict(topic=TOPIC, sources=summaries), f, indent=2)
print(f'[OK] {len(summaries)} sources scraped and summarized')

In [ ]:
# ESSAY OUTLINE + CHAPTERS
print('Generating outline...')
st = '\n\n'.join([('SOURCE: ' + s['source'] + '\n' + s['summary']) for s in summaries])
outline_prompt = 'Essay outline for ' + str(NARRATION_LENGTH) + 's video: "' + TOPIC + '"\n\nRESEARCH:\n' + st + '\n\nExactly ' + str(NUM_CHAPTERS) + ' chapters. JSON only: {"title":"...","thesis":"...","chapters":[{"chapter":1,"title":"...","summary":"...","key_points":["..."]}]}'
for attempt in range(3):
    try:
        outline = parse_json_response(query_llm(outline_prompt, 'Expert essay writer.'))
        break
    except Exception as e:
        print(f'  Retry {attempt+1}: {e}')

if len(outline.get('chapters', [])) > NUM_CHAPTERS:
    outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]

for i, ch in enumerate(outline['chapters']):
    print(f'Writing Ch {ch["chapter"]}: {ch["title"]}')
    sec_prompt = 'Write ' + str(SECTIONS_PER_CHAPTER) + ' sections. MAX ' + str(int(WORDS_PER_SCENE)) + ' words each.\nTOPIC: ' + TOPIC + '\nCHAPTER: ' + ch['title'] + '\nJSON: [{"section":1,"narration":"...","visual_direction":"...","mood":"..."}]'
    try:
        sections = parse_json_response(query_llm(sec_prompt, 'Scriptwriter. MAX ' + str(int(WORDS_PER_SCENE)) + ' words per section.'))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except:
        sections = [dict(section=1, narration=ch['title'], visual_direction='', mood='neutral')]
    outline['chapters'][i]['sections'] = sections[:SECTIONS_PER_CHAPTER]

outline['credits'] = dict(sources=[s['source'] for s in summaries])
with open(SESSION_DIR / 'essay_script.json', 'w') as f:
    json.dump(outline, f, indent=2)
tw = sum(len(s.get('narration','').split()) for c in outline['chapters'] for s in c.get('sections', []))
print(f'[OK] Essay: {outline.get("title","")} -- {tw} words')

In [ ]:
# STORYBOARD + STYLES
STYLE_ARCS = dict(
    calm_to_dramatic=dict(early=['documentary','nature'], middle=['news_report','sports_commentary'], late=['action_movie','horror']),
    documentary_journey=dict(early=['documentary'], middle=['nature','news_report'], late=['action_movie','sports_commentary']),
)
STYLES = dict(
    documentary='cinematic, professional lighting',
    nature='wide nature shots, golden hour',
    news_report='news studio, professional',
    sports_commentary='dynamic angles, slow motion',
    action_movie='dynamic movement, dramatic lighting',
    horror='dark lighting, shadows, fog',
)
CHOSEN_ARC = 'calm_to_dramatic'
arc = STYLE_ARCS[CHOSEN_ARC]

all_sec = []
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        d = dict(**sec)
        d['chapter_title'] = ch['title']
        d['chapter'] = ch.get('chapter', 0)
        all_sec.append(d)

scenes = []
sn = 1; si = 0
for stype in scene_types:
    if si < len(all_sec):
        sec = all_sec[si]
    else:
        sec = dict(narration='', visual_direction='', mood='neutral', chapter_title=TOPIC, section=si+1)
    scenes.append(dict(
        scene=sn, scene_type=stype,
        title=str(sec.get('chapter_title','')) + ' - Part ' + str(sec.get('section', si+1)),
        narration=sec.get('narration', ''),
        visual_direction=sec.get('visual_direction', ''),
        mood=sec.get('mood', ''),
        duration=SECONDS_PER_SCENE))
    sn += 1; si += 1

ai_sc = [s for s in scenes if s['scene_type'] == 'ai_generated']
total_ai = len(ai_sc)
ee = max(1, int(total_ai * 0.25))
ls = max(ee + 1, int(total_ai * 0.75))
last = None
for i, sc in enumerate(ai_sc):
    phase = 'early' if i < ee else ('late' if i >= ls else 'middle')
    pool = arc.get(phase, ['documentary'])
    avail = [s for s in pool if s != last] or pool
    st = random.choice(avail)
    sc['assigned_style'] = st
    sc['visual_keywords'] = STYLES.get(st, '')
    last = st

with open(SESSION_DIR / 'storyboard.json', 'w') as f:
    json.dump(scenes, f, indent=2)

print(f'Storyboard: {len(scenes)} scenes')
for s in scenes:
    tag = ' [' + s.get('assigned_style','') + ']' if s.get('assigned_style') else ''
    print(f'  Scene {s["scene"]}: [{s["scene_type"]}] {s["title"]}{tag}')

In [ ]:
# T2V PROMPTS (AI scenes only)
OPENINGS = ['Start with SHOT TYPE', 'Start with ACTION', 'Start with ENVIRONMENT']
prompts = []
ai_scenes_list = [s for s in scenes if s['scene_type'] == 'ai_generated']
if not ai_scenes_list:
    print('[!!] No AI scenes in plan! Check ARCHIVE_RATIO / UPLOAD_RATIO.')

for idx, sc in enumerate(ai_scenes_list):
    st = sc.get('assigned_style', 'documentary')
    vk = sc.get('visual_keywords', '')
    print(f'  Prompt {idx+1}/{len(ai_scenes_list)}: Scene {sc["scene"]} [{st}]')
    user_p = 'T2V prompt for ' + str(SECONDS_PER_SCENE) + 's:\nVisual: ' + sc.get('visual_direction','') + '\nMood: ' + sc.get('mood','') + '\n' + random.choice(OPENINGS) + '\nOutput ONLY the prompt text.'
    sys_p = 'T2V prompt expert. ONE paragraph, 3-5 sentences. Style: ' + st + '. Visual: ' + vk + '. No text overlays.'
    t2v = query_llm(user_p, sys_p).strip().strip('"')
    prompts.append(dict(scene=sc['scene'], title=sc['title'], t2v_prompt=t2v,
        style=st, narration=sc.get('narration',''), duration=SECONDS_PER_SCENE))

with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
print(f'[OK] {len(prompts)} T2V prompts generated')

---
# 03: Archive Scraper (optional)

Skip if ARCHIVE_RATIO = 0

In [ ]:
ARCHIVE_URLS = []
PREFER_HEIGHT = 480; MAX_CLIP_DURATION = 10; MIN_CLIP_DURATION = 3; MAX_EXTRACT_DURATION = 180

def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError('Cannot parse: ' + url)

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 0

archive_clips = []
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
if ARCHIVE_URLS and NUM_ARCHIVE_SCENES > 0:
    ATEMP = Path('/content/temp_archive'); ATEMP.mkdir(exist_ok=True)
    for i, url in enumerate(ARCHIVE_URLS):
        try: aid = parse_archive_id(url)
        except ValueError as e: print(f'Error: {e}'); continue
        print(f'Downloading [{i+1}/{len(ARCHIVE_URLS)}] {aid}')
        out_tpl = str(ATEMP / (aid + '.%(ext)s'))
        try: subprocess.run(['yt-dlp', 'https://archive.org/details/' + aid, '-f', 'bestvideo[height<=' + str(PREFER_HEIGHT) + ']+bestaudio/best[height<=' + str(PREFER_HEIGHT) + ']/best', '--merge-output-format', 'mp4', '--no-playlist', '--no-warnings', '-o', out_tpl, '--quiet'], check=True, timeout=600)
        except: print('  Download failed'); continue
        video = None
        for fp in ATEMP.iterdir():
            if fp.stem == aid and fp.suffix in ('.mp4','.mkv','.webm'): video = fp; break
        if not video: continue
        total = get_dur(video); clip_dir = ARCHIVE_DIR / aid; clip_dir.mkdir(exist_ok=True)
        extract_end = min(total, MAX_EXTRACT_DURATION) if MAX_EXTRACT_DURATION > 0 else total
        t = 0; idx2 = 0
        while t < extract_end:
            clip_dur = min(MAX_CLIP_DURATION, extract_end - t)
            if clip_dur < MIN_CLIP_DURATION: break
            clip_out = clip_dir / ('archive_' + f'{idx2:03d}' + '.mp4')
            r = subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',str(video),'-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','23','-an',str(clip_out)], capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                archive_clips.append(dict(path=str(clip_out), duration=round(clip_dur,1), archive_id=aid)); idx2 += 1
            t += clip_dur
        video.unlink(missing_ok=True); print(f'  {idx2} clips')
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f: json.dump(dict(clips=archive_clips, total=len(archive_clips)), f, indent=2)
    print(f'[OK] {len(archive_clips)} archive clips')
elif NUM_ARCHIVE_SCENES > 0: print(f'[!!] Need {NUM_ARCHIVE_SCENES} archive clips but no ARCHIVE_URLS!')
else: print('No archive footage needed')

---
# 04: Upload Footage (optional)

Skip if UPLOAD_RATIO = 0

In [ ]:
upload_clips = []
if NUM_UPLOAD_SCENES > 0:
    from google.colab import files as colab_files
    UPLOADS_DIR = SESSION_DIR / 'uploads'; UPLOADS_DIR.mkdir(exist_ok=True)
    print(f'Upload {NUM_UPLOAD_SCENES} MP4 clips:')
    uploaded = colab_files.upload()
    for fn, content in uploaded.items():
        dest = UPLOADS_DIR / fn
        with open(dest, 'wb') as f: f.write(content)
        upload_clips.append(dict(path=str(dest), filename=fn))
        print(f'  {fn} ({len(content)//1024}KB)')
    with open(SESSION_DIR / 'upload_clips.json', 'w') as f: json.dump(dict(clips=upload_clips, total=len(upload_clips)), f, indent=2)
else: print('No uploads needed (UPLOAD_RATIO = 0)')

---
# 05: Effects

In [ ]:
EFFECT_MODE = 'random'; EFFECT_INTENSITY = 0.7
if ENABLE_EFFECTS and prompts:
    enames = ['film_grain','vhs_artifacts','celluloid_scratches','sepia_tone','bw_transition','color_grade_warm','color_grade_cool','vignette','flicker','desaturate']
    for p in prompts:
        p['ffmpeg_effects'] = [random.choice(enames)]
        p['effect_intensity'] = EFFECT_INTENSITY
    with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
    for p in prompts: print(f'  Scene {p["scene"]}: {p["ffmpeg_effects"][0]}')
elif not prompts: print('No AI prompts for effects')
else: print('Effects disabled')

---
# 06: The Voice

In [ ]:
VOICES = dict(documentary='en-US-GuyNeural', female_us='en-US-AriaNeural', male_uk='en-GB-RyanNeural')
CHOSEN_VOICE = 'documentary'
voice_name = VOICES[CHOSEN_VOICE]

async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))

audio_files = []
narrations = [dict(scene=s['scene'], narration=s['narration']) for s in scenes if s.get('narration')]
for idx, n in enumerate(narrations):
    out = AUDIO_DIR / ('narration_' + f'{n["scene"]:03d}' + '.mp3')
    print(f'  TTS {idx+1}/{len(narrations)}: Scene {n["scene"]}')
    try:
        await gen_tts(n['narration'], out, voice_name)
        dur_val = get_dur(out) or 5.0
        audio_files.append(dict(scene=n['scene'], path=str(out), duration=dur_val, text=n['narration']))
    except Exception as e: print(f'    Error: {e}')

dur_map = {a['scene']: a['duration'] for a in audio_files}
for p in prompts:
    if p['scene'] in dur_map: p['duration'] = max(3, min(10, int(dur_map[p['scene']]) + 1))
with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
with open(SESSION_DIR / 'audio_manifest.json', 'w') as f: json.dump(dict(voice=voice_name, files=audio_files), f, indent=2)
td = sum(a['duration'] for a in audio_files)
print(f'[OK] {len(audio_files)} narrations, {td:.1f}s total')

---
# 07: AI Video Generation

In [ ]:
import fal_client
MODELS = {'wan-t2v': dict(endpoint='fal-ai/wan-t2v'), 'ltx-video': dict(endpoint='fal-ai/ltx-video/v2.1')}
CHOSEN_MODEL = 'wan-t2v'
model = MODELS[CHOSEN_MODEL]

if not prompts:
    print('No AI prompts. Nothing to generate.')
else:
    generated_videos = []
    t0 = _time.time()
    for i, pd in enumerate(prompts):
        sn = pd['scene']
        print(f'  Generating {i+1}/{len(prompts)}: Scene {sn} [{pd["style"]}]...')
        try:
            inp = dict(prompt=pd['t2v_prompt'], aspect_ratio='16:9', enable_prompt_expansion=False)
            result = fal_client.subscribe(model['endpoint'], arguments=inp)
            url = None
            if isinstance(result, dict):
                if 'video' in result and isinstance(result['video'], dict): url = result['video'].get('url')
                elif 'video' in result: url = result['video']
                elif 'url' in result: url = result['url']
            if url:
                vp = VIDEOS_DIR / ('scene_' + f'{sn:03d}' + '.mp4')
                with open(vp, 'wb') as f: f.write(requests.get(url, timeout=120).content)
                generated_videos.append(dict(scene=sn, path=str(vp), duration=pd.get('duration',5), source='generated'))
                print(f'    OK')
        except Exception as e: print(f'    Error: {e}')
    el = _time.time() - t0
    print(f'[OK] {len(generated_videos)}/{len(prompts)} videos in {el:.0f}s')
    with open(SESSION_DIR / 'video_manifest.json', 'w') as f: json.dump(dict(model=CHOSEN_MODEL, videos=generated_videos), f, indent=2)

---
# 08: Subtitles (optional)

Skip if ENABLE_SUBTITLES = False

In [ ]:
sub_file = SESSION_DIR / 'subtitles.ass'
if ENABLE_SUBTITLES:
    from faster_whisper import WhisperModel
    wm = WhisperModel('base', device='cpu', compute_type='int8')
    afs = sorted(AUDIO_DIR.glob('narration_*.mp3'))
    all_words = []; toff = 0.0
    for af in afs:
        segs, info = wm.transcribe(str(af), word_timestamps=True, language='en')
        for seg in segs:
            if seg.words:
                for w in seg.words: all_words.append(dict(word=w.word.strip(), start=w.start+toff, end=w.end+toff))
        try: fdur = float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(af)], capture_output=True, text=True, timeout=10).stdout.strip())
        except: fdur = 5.0
        toff += fdur
    W = 1280; H = 720; sc_f = H / 512
    sz = int(80 * sc_f); ol = int(7 * sc_f); mv = int(20 * sc_f)
    style_line = 'Style: TT,Impact,' + str(sz) + ',&H00FFFFFF,&H0000FFFF,&H00000000,&H00000000,-1,0,0,0,100,100,2,0,1,' + str(ol) + ',0,5,10,10,' + str(mv) + ',1'
    hdr = '[Script Info]\nTitle: ROTBOTS\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: ' + str(W) + '\nPlayResY: ' + str(H) + '\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n' + style_line + '\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
    def fmt(sec):
        h = int(sec // 3600); m = int((sec % 3600) // 60); s = int(sec % 60); cs = int((sec % 1) * 100)
        return str(h) + ':' + f'{m:02d}' + ':' + f'{s:02d}' + '.' + f'{cs:02d}'
    dlg = []
    for w in all_words:
        wd = w['word'].upper()
        dlg.append('Dialogue: 0,' + fmt(w['start']) + ',' + fmt(w['end']) + ',TT,,0,0,0,,' + wd)
    with open(sub_file, 'w', encoding='utf-8') as f: f.write(hdr + '\n'.join(dlg))
    print(f'[OK] subtitles.ass: {len(dlg)} words')
else: print('Subtitles disabled')

---
# 09: Assemble

In [ ]:
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 5.0

def ff(cmd, desc=''):
    if desc: print(f'   {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    if r.returncode == 0:
        if desc: print('OK')
        return True
    if desc: print('FAIL')
    if r.stderr: print('   ' + r.stderr[-200:])
    return False

def build_effect_filter(name, intensity=0.7):
    i = max(0.0, min(1.0, intensity))
    table = dict(
        film_grain='noise=alls=' + str(int(12*i)) + ':allf=t',
        vhs_artifacts='noise=alls=' + str(int(30*i)) + ':allf=t,rgbashift=rh=' + str(int(2*i)) + ':bh=' + str(int(-2*i)),
        celluloid_scratches='noise=c0s=' + str(int(15*i)) + ':c0f=t',
        color_grade_warm='eq=saturation=' + f'{1+0.1*i:.3f}' + ':brightness=' + f'{0.02*i:.3f}',
        color_grade_cool='eq=saturation=' + f'{1-0.1*i:.3f}' + ',colorbalance=bs=' + f'{0.12*i:.3f}',
        vignette='vignette=PI/4*' + f'{i:.3f}',
        flicker='noise=alls=' + str(int(8*i)) + ':allf=t',
        desaturate='eq=saturation=' + f'{0.5+0.5*(1-i):.3f}',
    )
    if name == 'sepia_tone':
        inv = 1 - i
        return 'colorchannelmixer=' + f'{inv+i*0.393:.3f}' + ':' + f'{i*0.769:.3f}' + ':' + f'{i*0.189:.3f}' + ':0:' + f'{i*0.349:.3f}' + ':' + f'{inv+i*0.686:.3f}' + ':' + f'{i*0.168:.3f}' + ':0:' + f'{i*0.272:.3f}' + ':' + f'{i*0.534:.3f}' + ':' + f'{inv+i*0.131:.3f}' + ':0'
    if name == 'bw_transition':
        inv = 1 - i
        return 'colorchannelmixer=' + f'{inv+i*0.3:.3f}' + ':' + f'{i*0.4:.3f}' + ':' + f'{i*0.3:.3f}' + ':0:' + f'{i*0.3:.3f}' + ':' + f'{inv+i*0.4:.3f}' + ':' + f'{i*0.3:.3f}' + ':0:' + f'{i*0.3:.3f}' + ':' + f'{i*0.4:.3f}' + ':' + f'{inv+i*0.3:.3f}' + ':0'
    return table.get(name)

In [ ]:
# BUILD SCENE SEQUENCE
effects_map = {}
for p in prompts:
    if p.get('ffmpeg_effects'): effects_map[int(p['scene'])] = p

print(f'Building {len(scenes)} scenes...')
norm = []; arc_idx = 0; upl_idx = 0

for sc in scenes:
    sn = sc['scene']; stype = sc['scene_type']
    out = TEMP / ('scene_' + f'{sn:03d}' + '.mp4')
    src = None
    if stype == 'ai_generated':
        c = VIDEOS_DIR / ('scene_' + f'{sn:03d}' + '.mp4')
        if c.exists(): src = c
        else: print(f'  [AI] Scene {sn}: MISSING'); continue
    elif stype == 'archive':
        if arc_idx < len(archive_clips): src = Path(archive_clips[arc_idx]['path']); arc_idx += 1
        if not src or not src.exists(): print(f'  [ARC] Scene {sn}: no clip'); continue
    elif stype == 'upload':
        if upl_idx < len(upload_clips): src = Path(upload_clips[upl_idx]['path']); upl_idx += 1
        if not src or not src.exists(): print(f'  [UPL] Scene {sn}: no clip'); continue
    vf = 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    if stype == 'ai_generated' and sn in effects_map:
        for eff in effects_map[sn].get('ffmpeg_effects', []):
            ef = build_effect_filter(eff, effects_map[sn].get('effect_intensity', 0.7))
            if ef: vf += ',' + ef
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an','-t',str(sc.get('duration',5)),str(out)], '[' + stype[:3].upper() + '] Scene ' + str(sn)):
        norm.append(out)

print(f'{len(norm)} scenes ready')

In [ ]:
# CREDITS + CONCAT + AUDIO + FINAL
credits_path = None
if ENABLE_CREDITS:
    essay = None
    ef = SESSION_DIR / 'essay_script.json'
    if ef.exists():
        with open(ef) as f: essay = json.load(f)
    title = essay.get('title', 'Untitled') if essay else 'Untitled'
    src_list = essay.get('credits', {}).get('sources', []) if essay else []
    lines = [title, '', 'Generated by ROTBOTS', ''] + [s[:60] for s in src_list] + ['', 'LI-MA TDA 2026']
    D = 8; LH = 42; spd = (720 + len(lines) * LH) / D
    flt = []
    for i, l in enumerate(lines):
        if not l: continue
        safe = l.replace(chr(39), chr(8217)).replace(':', chr(92) + ':')
        flt.append('drawtext=text=' + chr(39) + safe + chr(39) + ':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+' + str(i*LH) + '-' + f'{spd:.1f}' + '*t')
    credits_path = TEMP / 'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i','color=c=black:s=1280x720:d=' + str(D) + ':r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)], 'Credits')

# Concat
cf = TEMP / 'concat.txt'
with open(cf, 'w') as f:
    for v in norm: f.write("file '" + str(v) + "'\n")
    if credits_path and credits_path.exists(): f.write("file '" + str(credits_path) + "'\n")
concat_out = TEMP / 'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)], 'Concat')
video_duration = dur(concat_out)
print(f'  Video: {video_duration:.1f}s')

# Audio with padding
audio_out = TEMP / 'with_audio.mp4'
audio_files_sorted = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
if audio_files_sorted:
    acf = TEMP / 'ac.txt'
    with open(acf, 'w') as f:
        for a in audio_files_sorted: f.write("file '" + str(a) + "'\n")
    narr = TEMP / 'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)], 'Concat narration')
    narr_padded = TEMP / 'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narr),'-af','apad=whole_dur=' + str(video_duration),'-c:a','libmp3lame','-b:a','128k',str(narr_padded)], 'Pad audio')
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr_padded),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)], 'Merge audio')
else:
    shutil.copy2(str(concat_out), str(audio_out))

# Subtitles
final = SESSION_DIR / 'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    la = TEMP / 'subtitles.ass'; shutil.copy2(str(sub_file), str(la))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf','ass=' + str(la),'-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)], 'Burn subs'):
        shutil.copy2(str(audio_out), str(final))
else:
    shutil.copy2(str(audio_out), str(final))

if final.exists():
    sz = final.stat().st_size / (1024*1024)
    print(f'\n[OK] Final: {dur(final):.1f}s, {sz:.1f}MB')

---
## Watch

In [ ]:
essay = None
ef = SESSION_DIR / 'essay_script.json'
if ef.exists():
    with open(ef) as f: essay = json.load(f)
title = essay.get('title', 'Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown('# ' + title + '\n*Generated by ROTBOTS*'))
    try: display(Video(str(final), embed=True, width=720))
    except: print('Drive: rotbots_workshop/' + SESSION_NAME + '/final_video.mp4')

print('\nPipeline Summary')
ai_count = sum(1 for s in scenes if s['scene_type'] == 'ai_generated')
arc_count = sum(1 for s in scenes if s['scene_type'] == 'archive')
upl_count = sum(1 for s in scenes if s['scene_type'] == 'upload')
print(f'  AI: {ai_count} scenes ({len(prompts)} generated)')
if arc_count: print(f'  Archive: {arc_count} scenes')
if upl_count: print(f'  Upload: {upl_count} scenes')
print(f'  Narrations: {len(audio_files)}')
if final.exists(): print(f'  Final: {dur(final):.1f}s')

---
# Done

Every step: automated decisions. What does that mean?

---
*ROTBOTS -- LI-MA TDA 2026, Amsterdam*